In [2]:
import pandas as pd
from pprint import pprint
import json
import re
from typing import Dict, List, Any


In [37]:
ds_plannerStudy_628 = pd.read_json(
    "Original datasets/plannerStudy_628.json", lines=True
)

In [38]:
samples = []
for row in ds_plannerStudy_628.index:
    samples.append({f"index_{row}": ds_plannerStudy_628["output"][row]})

In [39]:
def reindex_samples(samples):
    """
    Reindex the dataset so each sample has a fresh sequential index.
    Input: list of dicts like [{'index_327': json_string}, ...]
    Output: list of dicts like [{'index_0': json_string}, {'index_1': json_string}, ...]
    """
    new_samples = []
    for new_idx, sample in enumerate(samples):
        # Each sample has only one key like "index_327"
        old_key = list(sample.keys())[0]
        value = sample[old_key]
        new_key = f"index_{new_idx}"
        new_samples.append({new_key: value})
    return new_samples

In [40]:
def remove_invalid_json(samples):
    """
    Remove invalid JSON samples from dataset and count them.

    Args:
        samples (list): List of dicts with JSON strings under keys like 'index_i'.

    Returns:
        tuple: (cleaned_samples, invalid_count)
    """
    cleaned_samples = []
    invalid_count = 0

    for i, sample in enumerate(samples):
        # Each sample has one key like "index_327"
        key = list(sample.keys())[0]
        raw = sample[key]

        try:
            parsed = json.loads(raw)  # Try parsing JSON
            # If valid, keep the sample
            cleaned_samples.append(sample)
        except json.JSONDecodeError:
            # If invalid, increment counter
            invalid_count += 1
            print(f"Invalid JSON at sample {i} (key={key})")
    cleaned_samples = reindex_samples(cleaned_samples)
    return cleaned_samples, invalid_count


samples, invalid_count = remove_invalid_json(samples)

Invalid JSON at sample 49 (key=index_49)


In [9]:
# Compile all your day-name regex formats
DAYNAME_PATTERNS = [
    re.compile(
        r"^(?:Section:\s*)?(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)(?:\s*-\s*(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday))?\s*\(Week\s*\d+\)(?::\s*)?(.*)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(?:Section:\s*)?(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)\s*\(Week\s*\d+\):\s*(.*)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(?:Section:\s*)?(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday),\s*Week\s*\d+(?::\s*)?(.*)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(?:Section:\s*)?(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)\s*-\s*Week\s*\d+(?::\s*)?(.*)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(?:Section:\s*)?Week\s*\d+:\s*(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)\s*(.*)$",
        re.IGNORECASE,
    ),
]


def contains_dayname_format(section_name: str) -> bool:
    """Check if a section name matches any of the day-name formats."""
    return any(p.match(section_name) for p in DAYNAME_PATTERNS)


def remove_dayname_plans(samples):
    filtered_samples = []

    for i, sample in enumerate(samples):
        try:
            sample_key = next(iter(sample.keys()))
        except StopIteration:
            print(f"Warning: Empty sample at position {i}. Skipping.")
            continue

        raw = sample[sample_key]

        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            print(f"Warning: Sample '{sample_key}' contains invalid JSON. Skipping.")
            continue

        sections = parsed.get("sections", [])
        has_dayname_format = any(
            contains_dayname_format(sec.get("name", "")) for sec in sections
        )

        if not has_dayname_format:
            filtered_samples.append(sample)
    filtered_samples=reindex_samples(filtered_samples)
    return filtered_samples



# print(f"original length:{len(samples)}")
# filtered_samples =remove_dayname_plans(samples)
# filtered_samples=reindex_samples(filtered_samples)
# print(f"filtered length:{len(filtered_samples)}")
# # pprint(samples[0])
# print("===========")
# pprint(filtered_samples[316])

In [70]:
def summarize_sections_with_regex(
    filtered_samples, match_regex=None, exclude_regex=None
):
    """
    Summarize and print plans that contain sections matching a given regex pattern.
    You can provide a match regex, an exclude regex, or both.

    Args:
        filtered_samples (list): List of dicts with JSON strings under keys like 'index_i'.
        match_regex (str, optional): Regex string to match section names.
        exclude_regex (str, optional): Regex string to exclude section names.
    """
    match_pattern = re.compile(match_regex, re.IGNORECASE) if match_regex else None
    exclude_pattern = (
        re.compile(exclude_regex, re.IGNORECASE) if exclude_regex else None
    )

    week_sections_summary = []
    indexing_error = 0

    for i, sample in enumerate(filtered_samples):
        key = f"index_{i}"
        try:
            raw = sample[key]
        except KeyError:
            indexing_error += 1
            print(f"Sample {i} does not have key '{key}'")
            continue

        try:
            parsed = json.loads(raw)  # convert JSON string → dict
        except json.JSONDecodeError as e:
            print(f"Error in sample {i}: {e}")
            print("Raw string:", raw[:200], "...")  # preview first 200 chars
            continue  # skip invalid JSON

        plan_name = parsed.get("name")
        sections = parsed.get("sections", [])

        # Apply match/exclude logic
        matched_sections = []
        for sec in sections:
            name = sec.get("name", "")
            match_ok = match_pattern.search(name) if match_pattern else True
            exclude_ok = not (exclude_pattern and exclude_pattern.search(name))
            if match_ok and exclude_ok:
                matched_sections.append(sec)

        if matched_sections:
            week_sections_summary.append(
                {
                    "index": i,
                    "plan_name": plan_name,
                    "count": len(matched_sections),
                    "sections": matched_sections,
                }
            )

    # Print results
    total_matched_sections = 0
    for summary in week_sections_summary:
        print(f"Plan: {summary['plan_name']}")
        print(f"Index: {summary['index']}")
        total_matched_sections += summary["count"]
        for sec in summary["sections"]:
            print(f"\tSection: {sec['name']}")
            print(f"\tOutline: {sec['outline']}\n")

    print(f"Total matched sections found: {total_matched_sections}")
    print(f"Total plans with matching sections: {len(week_sections_summary)}")
    print(f"Indexing errors: {indexing_error}")

In [11]:
DAY_PATTERN = re.compile(
    r"^(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday):\s*(.*)$",
    re.IGNORECASE,
)

def clean_day_colon_formats(samples):
    """
    Remove day names from section titles in the dataset.
    Keeps the real title and reindexes the dataset.
    ex: "Monday: Introduction" -> "Introduction"

    Args:
        samples (list): List of dicts with JSON strings under keys like 'index_i'.

    Returns:
        list: Cleaned and reindexed dataset.
    """
    new_samples = []

    for new_idx, sample in enumerate(samples):
        old_key = list(sample.keys())[0]
        raw = sample[old_key]

        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            # Skip invalid JSON
            continue

        cleaned_sections = []
        for sec in parsed.get("sections", []):
            name = sec.get("name", "")
            match = DAY_PATTERN.match(name)
            if match:
                
                # Group 2 = real title after day name
                real_title = match.group(2).strip()
                if not real_title:
                    real_title = "Study Task"
                sec["name"] = real_title
            cleaned_sections.append(sec)

        parsed["sections"] = cleaned_sections
        new_key = f"index_{new_idx}"
        new_samples.append({new_key: json.dumps(parsed)})
    
    return new_samples



In [15]:
# Regex patterns you specified
DAY_DASH_DAY_PATTERN = re.compile(
    r"^(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday) - Day\s*(.*)$",
    re.IGNORECASE,
)
DAY_DASH_PATTERN = re.compile(
    r"^(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday) - \s*(.*)$",
    re.IGNORECASE,
)


def clean_day_dash_formats(samples):
    """
    Clean section names that match day-name dash formats.
    Removes the day name and dash, keeps the real title.
    Reindexes the dataset sequentially.
    ex: "Monday - Day 1: Introduction" → "Introduction"

    Args:
        samples (list): List of dicts with JSON strings under keys like 'index_i'.

    Returns:
        list: Cleaned and reindexed dataset.
    """
    new_samples = []

    for new_idx, sample in enumerate(samples):
        old_key = list(sample.keys())[0]
        raw = sample[old_key]

        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            continue

        cleaned_sections = []
        for sec in parsed.get("sections", []):
            name = sec.get("name", "")
            if DAY_DASH_DAY_PATTERN.match(name):
                real_title = DAY_DASH_DAY_PATTERN.match(name).group(2).strip()
                sec["name"] = real_title if real_title else "Study Task"
            elif DAY_DASH_PATTERN.match(name):
                real_title = DAY_DASH_PATTERN.match(name).group(2).strip()
                sec["name"] = real_title if real_title else "Study Task"
            cleaned_sections.append(sec)

        parsed["sections"] = cleaned_sections
        new_key = f"index_{new_idx}"
        new_samples.append({new_key: json.dumps(parsed)})

    return new_samples

# print(len(filtered_samples))
# day_dash_cleaned=clean_day_dash_formats(filtered_samples)
# print(len(day_dash_cleaned))

In [72]:
def apply_ai_tasks_to_dataset(original_dataset: list, ai_results: list) -> list:
    """
    Replace section names in the original dataset with AI-processed task names.

    Args:
        original_dataset (list): List of dicts like {'index_599': '...json string...'}
        ai_results (list): List of dicts like {"plan_index": 599, "section_index": 6, "task": "..."}

    Returns:
        list: Updated dataset with replaced section names.
    """
    # Build a lookup for faster access
    ai_lookup = {}
    for task in ai_results:
        ai_lookup.setdefault(task["plan_index"], []).append(task)

    updated_dataset = []
    for sample in original_dataset:
        key = list(sample.keys())[0]  # e.g. "index_599"
        plan_index = int(key.split("_")[1])

        parsed = json.loads(sample[key])

        if plan_index in ai_lookup:
            for task in ai_lookup[plan_index]:
                section_idx = task["section_index"]
                if section_idx < len(parsed["sections"]):
                    parsed["sections"][section_idx]["name"] = task["task"]

        # Save back
        updated_dataset.append({key: json.dumps(parsed, ensure_ascii=False)})

    return updated_dataset

In [90]:
def deduplicate_dataset(input_file: str, output_file: str) -> None:
    """
    Remove duplicate rows from a dataset, keeping the first occurrence.

    Args:
        input_file (str): Path to the checkpoint JSON file.
        output_file (str): Path to save the deduplicated dataset.
    """
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)  # list of dicts

    seen = set()
    deduped = []
    for row in data:
        # Use (plan_index, section_index) as unique key
        key = (row["plan_index"], row["section_index"])
        if key not in seen:
            seen.add(key)
            deduped.append(row)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(deduped, f, indent=2, ensure_ascii=False)

    print(f"✅ Deduplicated dataset saved to {output_file}")
    print(f"Original size: {len(data)}, Deduplicated size: {len(deduped)}")

In [91]:
deduplicate_dataset(
    "stage2/ai_processed_tasks_names.json", "stage2/ai_processed_tasks_names.json"
)

✅ Deduplicated dataset saved to stage2/ai_processed_tasks_names.json
Original size: 1801, Deduplicated size: 1801


In [81]:
filtered=remove_dayname_plans(samples)
filtered=clean_day_colon_formats(filtered)
filtered=clean_day_dash_formats(filtered)
len(filtered)

602

In [92]:
with open("stage2/ai_processed_tasks_names.json", "r", encoding="utf-8") as f:
    ai_processed = json.load(f)

In [80]:
ai_processed[0]

{'plan_index': 3,
 'section_index': 1,
 'task': 'Read article on microplastics, review flashcards, and take quiz'}

In [88]:
filtered=apply_ai_tasks_to_dataset(filtered,ai_processed)

In [79]:
pprint(samples[3]["index_3"])

('{\n'
 '  "name": "AP Environmental Science: Microplastic Pollution and Marine '
 'Ecosystems Study Plan",\n'
 '  "description": "This study plan is designed to help Kelly Parker prepare '
 'for the AP Environmental Science test on the impact of microplastic '
 'pollution on marine trophic dynamics and coral reef health. The plan is '
 "structured to accommodate Kelly's busy schedule and medium learning pace, "
 'focusing on key concepts and effective study strategies.",\n'
 '  "sections": [\n'
 '    {\n'
 '      "name": "Tuesday Evening",\n'
 '      "outline": "Begin by watching an introductory video on microplastic '
 'pollution and its effects on marine ecosystems. Use the Pomodoro technique: '
 'study for 25 minutes, then take a 5-minute break. After watching the video, '
 "review flashcards on key terms such as 'microplastics,' 'trophic dynamics,' "
 'and \'coral reef health.\'",\n'
 '      "resources": [\n'
 '        {\n'
 '          "type": "video",\n'
 '          "keywords": [

In [89]:
summarize_sections_with_regex(filtered,match_regex=r"(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)")

Total matched sections found: 0
Total plans with matching sections: 0
Indexing errors: 0


In [74]:
summarize_sections_with_regex(
    filtered, match_regex=r"(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)"
)

Plan: AP Environmental Science: Microplastic Pollution and Marine Ecosystems Study Plan
Index: 3
	Section: Read article on microplastics, review Tuesday flashcards, take quiz
	Outline: Read a detailed article on the sources and types of microplastics. Use spaced repetition to review the flashcards from Tuesday. Then, take a quiz to test your understanding of the material covered so far.

Plan: AP Environmental Science: Biogeochemical Cycles Study Plan
Index: 9
	Section: Review flashcards and take short quiz on Monday
	Outline: Since Monday is a busy day, focus on a quick review session. Go through flashcards and take a short quiz to reinforce your knowledge. Use spaced repetition to ensure retention.

Plan: Intensive Study Plan for Unit 4: Microbial Interactions and Bioremediation Techniques in Polluted Aquatic Ecosystems
Index: 32
	Section: Create a mind map of microbial interactions and bioremediation techniques from Wednesday's notes (1 hour)
	Outline: Review your notes from Wednesd

In [57]:
pprint(json.loads(filtered[599]["index_599"]))

{'description': 'A detailed 6-day study plan to help Justin Martin prepare for '
                "the test on 'The Columbian Exchange and Its Global Impacts: "
                'Analyzing the Ecological, Economic, and Cultural '
                "Transformations from 1492 to 1750'.",
 'name': 'AP Modern World History: The Columbian Exchange Study Plan',
 'sections': [{'name': 'Study Columbian Exchange: Watch video, take notes '
                       '(Pomodoro)',
               'outline': 'Start with an overview of the Columbian Exchange. '
                          'Watch a video that explains the basic concepts and '
                          'significance of the Columbian Exchange. Take notes '
                          'on key points such as the major goods exchanged, '
                          'the impact on indigenous populations, and the '
                          'economic changes. Use the Pomodoro technique to '
                          'stay focused: 25 minutes of study fol

In [69]:
def extract_sections_by_regex(filtered_samples, regex_pattern):
    """
    Filter sections by regex and output one-line JSON dataset
    with plan index, section order index, and outline.

    Args:
        filtered_samples (list): List of dicts with JSON strings under keys like 'index_i'.
        regex_pattern (str): Regex string to match section names.

    Returns:
        list: List of JSON strings, each one line.
    """
    compiled_pattern = re.compile(regex_pattern, re.IGNORECASE)
    results = []

    for i, sample in enumerate(filtered_samples):
        key = f"index_{i}"
        if key not in sample:
            continue

        try:
            parsed = json.loads(sample[key])
        except json.JSONDecodeError:
            continue

        sections = parsed.get("sections", [])
        for j, sec in enumerate(sections):
            name = sec.get("name", "")
            outline = sec.get("outline", "")
            if compiled_pattern.search(name):
                record = {"plan_index": i, "section_index": j, "outline": outline}
                # Save as one-line JSON string
                results.append(json.dumps(record))

    return results

preLLM_dataset = extract_sections_by_regex(filtered, r"(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)")
# Save as JSONL (one JSON object per line)
with open("stage2/outlines to summarize.json", "w", encoding="utf-8") as f:
    for record in preLLM_dataset:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
import re
import json


def save_filtered_sections(
    filtered_samples, regex_pattern, output_file="filtered_dataset.jsonl"
):
    """
    Filter sections by regex and save results as a JSONL dataset.
    Each line contains: plan_index, section_index, outline.

    Args:
        filtered_samples (list): List of dicts with JSON strings under keys like 'index_i'.
        regex_pattern (str): Regex string to match section names.
        output_file (str): Path to save JSONL dataset.
    """
    compiled_pattern = re.compile(regex_pattern, re.IGNORECASE)
    results = []

    for i, sample in enumerate(filtered_samples):
        key = f"index_{i}"
        if key not in sample:
            continue

        try:
            parsed = json.loads(sample[key])
        except json.JSONDecodeError:
            continue

        sections = parsed.get("sections", [])
        for j, sec in enumerate(sections):
            name = sec.get("name", "")
            outline = sec.get("outline", "")
            if compiled_pattern.search(name):
                record = {"plan_index": i, "section_index": j, "outline": outline}
                results.append(record)

    # Save as JSONL (one JSON object per line)
    with open(output_file, "w", encoding="utf-8") as f:
        for record in results:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Saved {len(results)} records to {output_file}")

In [ ]:
def expand_week_sections(plan: dict) -> dict:
    """
    Replace sections with names like 'Week 1: ...' into 5 daily tasks + 2 break days.
    Returns a new plan dict with expanded sections.
    """
    new_sections = []
    try:
        plan = json.loads(plan)
    except json.JSONDecodeError:
        print("Failed to parse plan as JSON")
        return plan  # Return original plan if parsing fails
    # print(plan.values())
    # plan=plan.values()
    for sec in plan.get("sections", []):
        try:
            name = sec.get("name", "")
            print(name)
        except Exception as e:
            print(
                f"Error processing section in plan {plan.get('name', 'Unknown')}: {e}"
            )
            continue
        match = re.match(r"(Week\s*\d+):\s*(.*)", name, flags=re.IGNORECASE)
        if match:
            week_label, base_title = match.groups()
            # Create 5 study days
            for i in range(1, 6):
                new_sections.append(
                    {
                        "index": i - 1,
                        "task": f"{base_title} - Day {i}",
                        "description": sec.get("outline", ""),
                        "is_repeatable": False,
                        "repeat_pattern": "none",
                        "estimated_duration_minutes": {"min": 30, "max": 45},
                        "gap_flag": False,
                    }
                )
            # Add 2 break days
            for i in range(1, 3):
                new_sections.append(
                    {
                        "index": i - 1,
                        "task": f"Break Day",
                        "description": "Take a rest day to recharge and consolidate learning.",
                        "is_repeatable": False,
                        "repeat_pattern": "none",
                        "estimated_duration_minutes": {"min": 0, "max": 0},
                        "gap_flag": True,
                    }
                )
        else:
            # Keep non-week sections unchanged
            new_sections.append(sec)

    # Return updated plan
    return {**plan, "sections": new_sections}


s = expand_week_sections(samples[624]["index_624"])
pprint([sec.get("task") for sec in s.get("sections", [])])

In [ ]:
import json
import pandas as pd


def convert_row_to_schema(row):
    """
    Convert a single row of study plan data into the target schema.
    Assumes row contains keys like: course, unit, timeuntiltest, difficulty, busylevels, etc.
    """
    # Build baseline constraints
    baseline = {
        "fixed_constraints": f"Test scheduled in {row.get('timeuntiltest', 'N/A')}",
        "physical_constraints": f"Busy schedule: {row.get('busylevels', 'N/A')}",
        "non_negotiables": f"Must cover all unit topics for {row.get('unit', '')}",
    }

    # Build tasks from sections (if provided)
    tasks = []
    if "sections" in row:
        for i, section in enumerate(row["sections"]):
            tasks.append(
                {
                    "index": i,
                    "task": section.get("name", ""),
                    "description": section.get("outline", ""),
                    "is_repeatable": False,
                    "repeat_pattern": "none",
                    "estimated_duration_minutes": {
                        "min": 45,  # default estimate
                        "max": 90,  # default estimate
                    },
                }
            )

    schema = {
        "goal": f"Prepare for {row.get('course', '')} test on {row.get('unit', '')}",
        "goal_category": "one_time",
        "goal_type": "academic study",
        "time_horizon": row.get("timeuntiltest", ""),
        "description": f"A study plan for {row.get('course', '')} focusing on {row.get('unit', '')}.",
        "baseline": baseline,
        "tasks": tasks,
    }

    return schema


# Example: converting a dataset of 1k rows
def convert_dataset(input_file, output_file):
    """
    Reads a JSON or CSV dataset of study plans and converts each row.
    """
    # Load dataset (assuming JSON lines or CSV)
    if input_file.endswith(".json"):
        with open(input_file, "r") as f:
            data = json.load(f)
    else:
        df = pd.read_csv(input_file)
        data = df.to_dict(orient="records")

    # Convert each row
    converted = [convert_row_to_schema(row) for row in data]

    # Save output
    with open(output_file, "w") as f:
        json.dump(converted, f, indent=2)


# Example usage:
# convert_dataset("study_plans.json", "converted_schema.json")
# convert_dataset("study_plans.csv", "converted_schema.json")